# Resizer CPU Benchmark — Image Resizer
Resize 4K (3840x2160) → 1080p (1920x1080) on **ARM CPU (Cortex-A53)**.

**Run from Jupyter** (`http://<board-ip>:9090/lab`)

> ⚠️ Always run from Jupyter or pynq venv — system Python has NumPy version conflict with cv2.

Expected: **~377ms per frame, ~2.65 FPS, ~3.34W, ~0.79 FPS/Watt**

In [ ]:
import os, time, numpy as np
from PIL import Image

POWER_SENSOR = "/sys/class/hwmon/hwmon2/power1_input"
N_WARMUP = 3
N_RUNS   = 7
IN_SIZE  = (3840, 2160)
OUT_SIZE = (1920, 1080)

def read_power_mw():
    with open(POWER_SENSOR) as f:
        return int(f.read().strip()) / 1000

print(f"Current power: {read_power_mw()/1000:.2f} W (idle)")

## Step 1 — Create Test Image (4K)

In [ ]:
img = Image.fromarray(
    np.random.randint(0, 255, (IN_SIZE[1], IN_SIZE[0], 3), dtype=np.uint8))
print(f"Input:  {img.size[0]}x{img.size[1]} pixels (4K)")
print(f"Output: {OUT_SIZE[0]}x{OUT_SIZE[1]} pixels (1080p)")

## Step 2 — Warmup

In [ ]:
for _ in range(N_WARMUP):
    img.resize(OUT_SIZE, Image.BICUBIC)
print("Warmup done!")

## Step 3 — Benchmark with Power

In [ ]:
times_ms = []
power_samples = []
for i in range(N_RUNS):
    p_before = read_power_mw()
    t0 = time.time()
    img.resize(OUT_SIZE, Image.BICUBIC)
    t1 = time.time()
    p_after = read_power_mw()
    times_ms.append((t1 - t0) * 1000)
    power_samples.append((p_before + p_after) / 2)
    print(f"  Run {i+1}: {times_ms[-1]:.1f} ms, {power_samples[-1]/1000:.2f} W")

## Step 4 — Results

In [ ]:
avg_ms = sum(times_ms) / len(times_ms)
avg_power_w = sum(power_samples) / len(power_samples) / 1000
fps = 1000 / avg_ms
fps_per_watt = fps / avg_power_w

print("=" * 40)
print("CPU RESIZER RESULTS")
print("=" * 40)
print(f"Time/frame:  {avg_ms:.1f} ms")
print(f"FPS:         {fps:.2f}")
print(f"Power:       {avg_power_w:.2f} W")
print(f"FPS/Watt:    {fps_per_watt:.2f}")
print("=" * 40)
print(f"Expected:    ~377ms, ~2.65 FPS, ~3.34W, ~0.79 FPS/Watt")